# Financial Dashboard
**Comprehensive Credit Research Dashboard** | Universal template

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────────
COMPANY_ID = 'rusal'
STRESS_SCENARIOS = ['aluminium_downturn', 'usd_rub_shock', 'rate_spike']
HIST_START = 2014
EXPORT_HTML = True

# Color palette (Bloomberg-style)
C = {
    'hist': '#4e79a7', 'base': '#59a14f', 'fcst': '#76b7b2',
    's1': '#e15759', 's2': '#f28e2b', 's3': '#9c755f',
    'rev': '#4e79a7', 'ebitda': '#59a14f', 'ni': '#edc948',
    'debt': '#e15759', 'equity': '#76b7b2', 'cash': '#f28e2b',
    'cfo': '#59a14f', 'cfi': '#e15759', 'cff': '#4e79a7',
    'pos': '#59a14f', 'neg': '#e15759', 'neutral': '#bab0ac',
    'header': '#343a40', 'bg': '#fafafa', 'card': '#ffffff',
}

# ── IMPORTS ─────────────────────────────────────────────────────────────────────
import warnings, logging, sys
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)
from pathlib import Path as P

# Find project root
for _p in P('.').absolute().parents:
    if (_p / 'data_mart_v2.db').exists():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from IPython.display import display, HTML
except ImportError:
    display = lambda *a, **kw: None
    class HTML:
        def __init__(self, *a, **kw):
            pass

from engine.database.repository import Repository
from engine.orchestrator import build_model
from engine.rating.core import RatingEngine, RatingConfig, CreditMetrics
from engine import DB_PATH

FIGS = []  # collect plotly figures for HTML export
print('Config OK | ' + COMPANY_ID.upper() + ' | DB: ' + str(DB_PATH))

---
## Data Loading

In [ ]:
# ── LOAD ALL DATA ───────────────────────────────────────────────────────────────
with Repository(DB_PATH) as repo:
    D = {'is': {}, 'bs': {}, 'cf': {}}
    for stmt in ('is', 'bs', 'cf'):
        for yr in range(2011, 2036):
            row = repo.get_history_year(COMPANY_ID, stmt.upper(), yr)
            if row and any(v for v in row.values() if v):
                D[stmt][yr] = row

    # Debt instruments
    D['di'] = repo.query(
        'SELECT * FROM debt_instruments WHERE company_id=? ORDER BY opening_balance DESC',
        (COMPANY_ID,))

    # Macro factors (history + forecast)
    D['macro'] = repo.query(
        'SELECT factor_name, year, value FROM macro_factors ORDER BY factor_name, year')
    D['macro_fc'] = repo.query(
        'SELECT factor_name, year, value FROM macro_forecasts WHERE company_id=? '
        'ORDER BY factor_name, year', (COMPANY_ID,))

    # Production KPIs
    D['kpi'] = repo.query(
        "SELECT metric_name, year, value FROM preprocess_metrics "
        "WHERE company_id=? AND metric_group='production_kpi' AND year > 0 "
        "ORDER BY metric_name, year", (COMPANY_ID,))

    # Segments
    D['segs'] = repo.query(
        "SELECT sd.segment_name, sd.metric, p.year, sd.value "
        "FROM segment_data sd JOIN periods p ON sd.period_id=p.period_id "
        "WHERE sd.company_id=? ORDER BY p.year", (COMPANY_ID,))

# ── MODEL FORECAST ──────────────────────────────────────────────────────────────
result = build_model(
    COMPANY_ID,
    run_preprocessor=False,
    run_model=True,
    run_stress=True,
    stress_scenarios=STRESS_SCENARIOS)

D['fc'] = result.model_result.years          # Dict[int, YearState]
D['stress'] = result.stress_results or {}    # Dict[str, StressResult]

# ── RATINGS ─────────────────────────────────────────────────────────────────────
eng = RatingEngine(RatingConfig(methodology='sp'))
D['ratings'] = {}
for yr, st in D['fc'].items():
    try:
        D['ratings'][yr] = eng.calculate(CreditMetrics.from_year_state(st, yr))
    except Exception:
        pass

LAST_HIST = max(D['is'].keys())
print('Data loaded: '
      + str(len(D['is'])) + ' yr IS, '
      + str(len(D['fc'])) + ' yr fcst, '
      + str(len(D['di'])) + ' DI, '
      + str(len(D['stress'])) + ' stress, '
      + str(len(D['ratings'])) + ' ratings')

---
## Utils

In [ ]:
# ── HELPER FUNCTIONS ─────────────────────────────────────────────────────────────

def hy(start=HIST_START):
    """History years with revenue data."""
    return [y for y in range(start, LAST_HIST + 1) if D['is'].get(y, {}).get('revenue')]

def fy():
    """Forecast years."""
    return sorted(D['fc'].keys())

def ay():
    """All years (history + forecast)."""
    return sorted(set(hy()) | set(fy()))

def gi(yr, metric, stmt='is'):
    """Get indicator: works for both history dicts and forecast YearState objects."""
    if yr <= LAST_HIST:
        return D[stmt].get(yr, {}).get(metric, 0) or 0
    st = D['fc'].get(yr)
    if st is None:
        return 0
    return getattr(st, metric, 0) or 0

# Formatting helpers
M = lambda v: (v or 0) / 1e6
B = lambda v: (v or 0) / 1e9
pct = lambda a, b: a / b * 100 if b else 0.0
sdiv = lambda a, b: a / b if b else 0.0

def fmt_m(v):
    """Format as $XM."""
    return '${:,.0f}M'.format(M(v))

def fmt_pct(v):
    """Format as X.X%."""
    return '{:.1f}%'.format(v)

def fmt_x(v):
    """Format as X.Xx."""
    return '{:.1f}x'.format(v)

# Chart helpers
def vline(fig):
    """Add history/forecast separator line."""
    fig.add_vline(x=LAST_HIST + 0.5, line_dash='dash', line_color='gray', opacity=0.4)

def sty(fig, title='', h=420):
    """Apply standard styling to a plotly figure."""
    fig.update_layout(
        title='<b>' + title + '</b>',
        height=h,
        paper_bgcolor='white',
        plot_bgcolor=C['bg'],
        font=dict(family='Arial', size=11),
        hovermode='x unified',
        legend=dict(orientation='h', y=1.05, font=dict(size=10)),
        margin=dict(l=55, r=15, t=55, b=35))
    return fig

def add_fig(fig):
    """Append figure to FIGS list and show it."""
    FIGS.append(fig)
    fig.show()

# ── HTML TABLE BUILDER ──────────────────────────────────────────────────────────
def html_table(headers, rows, title='', widths=None):
    """Build a styled HTML table string."""
    s = '<div style="overflow-x:auto;margin:8px 0">'
    if title:
        s += '<h4 style="margin:4px 0;color:#343a40">' + title + '</h4>'
    s += '<table style="border-collapse:collapse;font-family:Arial;font-size:11px;width:100%">'
    s += '<thead><tr>'
    for i, h in enumerate(headers):
        w = ('width:' + widths[i]) if widths and i < len(widths) else ''
        s += ('<th style="background:#343a40;color:white;padding:6px 10px;text-align:center;'
              + w + '">' + str(h) + '</th>')
    s += '</tr></thead><tbody>'
    for ri, row in enumerate(rows):
        bg = '#ffffff' if ri % 2 == 0 else '#f8f9fa'
        s += '<tr style="background:' + bg + '">'
        for ci, val in enumerate(row):
            align = 'left' if ci == 0 else 'right'
            color = ''
            sv = str(val)
            if sv.startswith('-') or sv.startswith('($'):
                color = 'color:#e15759;'
            s += ('<td style="padding:5px 10px;border-bottom:1px solid #eee;text-align:'
                  + align + ';' + color + '">' + sv + '</td>')
        s += '</tr>'
    s += '</tbody></table></div>'
    return s

# ── STRESS VALUE ACCESSOR ──────────────────────────────────────────────────────
def sv_get(yr, metric, sr=None):
    """Get metric value from base forecast or stress result."""
    if sr is not None:
        svd = getattr(sr, 'stress_values', {}) or {}
        src = svd.get(yr, {})
        if isinstance(src, dict):
            return src.get(metric, 0) or 0
        return getattr(src, metric, 0) or 0
    st = D['fc'].get(yr)
    if st is None:
        return 0
    return getattr(st, metric, 0) or 0

def sv_nd_ebitda(yr, sr=None):
    """Net Debt / EBITDA from stress or base."""
    td = abs(sv_get(yr, 'long_term_debt', sr)) + abs(sv_get(yr, 'short_term_debt', sr))
    nd = td - (sv_get(yr, 'cash', sr) or 0)
    eb = sv_get(yr, 'ebitda', sr) or 1
    return sdiv(nd, eb)

# ── COVENANT HELPERS ────────────────────────────────────────────────────────────
COVENANTS = {
    'ND/EBITDA': {'max': 4.5, 'warning': 3.5},
    'ICR': {'min': 2.0, 'warning': 3.0},
    'DSCR': {'min': 1.2, 'warning': 1.5},
}

def cov_color(metric, value):
    """Return color based on covenant threshold."""
    spec = COVENANTS.get(metric, {})
    if 'max' in spec:
        if value > spec['max']:
            return C['neg']
        if value > spec['warning']:
            return C['s2']
        return C['pos']
    if 'min' in spec:
        if value < spec['min']:
            return C['neg']
        if value < spec['warning']:
            return C['s2']
        return C['pos']
    return ''

# ── RATING SCALE HELPERS ───────────────────────────────────────────────────────
RATING_SCALE = [
    'AAA', 'AA+', 'AA', 'AA-', 'A+', 'A', 'A-',
    'BBB+', 'BBB', 'BBB-', 'BB+', 'BB', 'BB-',
    'B+', 'B', 'B-', 'CCC+', 'CCC', 'CC', 'C', 'D']

def rn(r):
    """Rating to numeric (higher = better)."""
    try:
        return len(RATING_SCALE) - RATING_SCALE.index(r)
    except ValueError:
        return 0

def rating_color(r):
    """Color code for rating badge."""
    try:
        idx = RATING_SCALE.index(r)
    except ValueError:
        return '#888'
    if idx <= 9:
        return '#28a745'  # IG
    if idx <= 15:
        return '#fd7e14'  # BB/B
    return '#dc3545'  # CCC and below

print('Utils ready | ' + str(len(ay())) + ' total years')

---
## Section 0: Report Header

In [ ]:
# ── KPI CARDS ────────────────────────────────────────────────────────────────────
yr = LAST_HIST
d = D['is'].get(yr, {})
b = D['bs'].get(yr, {})

rev = d.get('revenue', 0) or 0
eb = d.get('ebitda', 0) or d.get('ebit', 0) or 0
ni = d.get('net_income', 0) or 0
td = abs(b.get('long_term_debt', 0) or 0) + abs(b.get('short_term_debt', 0) or 0)
nd = td - (b.get('cash', 0) or 0)
ta = b.get('total_assets', 0) or 0

# First forecast year rating
fyr = min(fy()) if fy() else yr + 1
rat = D['ratings'].get(fyr, {}).get('rating', '-')
rat_score = D['ratings'].get(fyr, {}).get('score', 0)
rc = rating_color(rat)

cards = [
    ('Revenue ' + str(yr), fmt_m(rev), C['rev']),
    ('EBITDA', fmt_m(eb) + ' (' + fmt_pct(pct(eb, rev)) + ')', C['ebitda']),
    ('Net Income', fmt_m(ni), C['ni']),
    ('Total Assets', fmt_m(ta), C['neutral']),
    ('Net Debt', fmt_m(nd), C['debt']),
    ('ND/EBITDA', fmt_x(sdiv(nd, eb)), C['debt']),
    ('Rating ' + str(fyr), rat + ' (' + '{:.0f}'.format(rat_score) + 'pts)', rc),
    ('Stress Scenarios', str(len(D['stress'])), C['s1']),
]

card_html = ''
for label, value, color in cards:
    card_html += (
        '<div style="background:' + C['card'] + ';border:1px solid #ddd;border-radius:8px;'
        'padding:14px 18px;min-width:130px;border-top:3px solid ' + color + '">'
        '<div style="font-size:10px;color:#888;text-transform:uppercase;letter-spacing:0.5px">'
        + label + '</div>'
        '<div style="font-size:20px;font-weight:700;color:#343a40;margin-top:4px">'
        + value + '</div></div>')

header_html = (
    '<div style="background:linear-gradient(135deg,#343a40,#495057);color:white;'
    'padding:20px 24px;border-radius:10px;margin-bottom:16px">'
    '<h1 style="margin:0;font-size:28px">' + COMPANY_ID.upper()
    + ' | Financial Dashboard</h1>'
    '<p style="margin:4px 0 0;opacity:0.7">Credit Research Report | '
    + str(min(hy())) + '-' + str(max(fy())) + '</p></div>'
    '<div style="display:flex;gap:12px;margin:12px 0;flex-wrap:wrap">'
    + card_html + '</div>')

try:
    display(HTML(header_html))
except Exception:
    print(COMPANY_ID.upper() + ' Dashboard | Rev: ' + fmt_m(rev) + ' | Rating: ' + rat)

---
## Section 1: Macro Environment

In [ ]:
# ── MACRO DASHBOARD ──────────────────────────────────────────────────────────────
macro_h = {}
for row in D['macro']:
    fn = row['factor_name']
    if fn not in macro_h:
        macro_h[fn] = {}
    macro_h[fn][row['year']] = row['value']

macro_f = {}
for row in D['macro_fc']:
    fn = row['factor_name']
    if fn not in macro_f:
        macro_f[fn] = {}
    macro_f[fn][row['year']] = row['value']

# Identify key macro factors
al_keys = [k for k in list(macro_h.keys()) + list(macro_f.keys())
           if 'alumin' in k.lower() or 'lme' in k.lower() or 'al_' in k.lower()]
fx_keys = [k for k in list(macro_h.keys()) + list(macro_f.keys())
           if 'usd' in k.lower() or 'rub' in k.lower() or 'fx' in k.lower()]
rate_keys = [k for k in list(macro_h.keys()) + list(macro_f.keys())
             if 'rate' in k.lower() or 'cbr' in k.lower() or 'key_rate' in k.lower()]

all_macro_keys = sorted(set(list(macro_h.keys()) + list(macro_f.keys())))

# Create 2x1 subplot
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Commodity Forward Curves', 'FX & Rates'],
    vertical_spacing=0.15)

colors_cycle = [C['rev'], C['ebitda'], C['s1'], C['s2'], C['s3'], C['ni']]
plotted = 0

# Plot commodity factors
commodity_keys = al_keys if al_keys else all_macro_keys[:3]
for i, k in enumerate(sorted(set(commodity_keys))[:3]):
    merged = {}
    merged.update(macro_h.get(k, {}))
    merged.update(macro_f.get(k, {}))
    if not merged:
        continue
    yrs_k = sorted(merged.keys())
    vals_k = [merged[y] for y in yrs_k]
    cc = colors_cycle[i % len(colors_cycle)]
    fig.add_trace(go.Scatter(
        x=yrs_k, y=vals_k, name=k.replace('_', ' ').title(),
        line=dict(color=cc, width=2)), row=1, col=1)
    plotted += 1

# Plot FX/rate factors
fx_rate_keys = (fx_keys + rate_keys) if (fx_keys or rate_keys) else all_macro_keys[3:6]
for i, k in enumerate(sorted(set(fx_rate_keys))[:3]):
    merged = {}
    merged.update(macro_h.get(k, {}))
    merged.update(macro_f.get(k, {}))
    if not merged:
        continue
    yrs_k = sorted(merged.keys())
    vals_k = [merged[y] for y in yrs_k]
    cc = colors_cycle[(i + 3) % len(colors_cycle)]
    fig.add_trace(go.Scatter(
        x=yrs_k, y=vals_k, name=k.replace('_', ' ').title(),
        line=dict(color=cc, width=2, dash='dash')), row=2, col=1)
    plotted += 1

if plotted > 0:
    vline(fig)
    add_fig(sty(fig, 'Macro Assumptions: ' + COMPANY_ID.upper(), 520))
else:
    print('No macro data available')

# HTML macro assumptions table
if all_macro_keys:
    fyrs = sorted(set(y for d in macro_f.values() for y in d))[:6]
    if fyrs:
        m_headers = ['Factor'] + [str(y) for y in fyrs]
        m_rows = []
        for k in all_macro_keys[:15]:
            row_v = [k.replace('_', ' ').title()]
            for y in fyrs:
                v = macro_f.get(k, {}).get(y, macro_h.get(k, {}).get(y))
                row_v.append('{:,.1f}'.format(v) if v is not None else '-')
            m_rows.append(row_v)
        try:
            display(HTML(html_table(m_headers, m_rows, 'Macro Forecasts')))
        except Exception:
            print('Macro factors: ' + ', '.join(all_macro_keys[:10]))

---
## Section 2: Industry & Production KPIs

In [ ]:
# ── PRODUCTION KPIs ──────────────────────────────────────────────────────────────
kpi_data = {}
for row in D['kpi']:
    mn = row['metric_name']
    if mn not in kpi_data:
        kpi_data[mn] = {}
    kpi_data[mn][row['year']] = row['value']

if kpi_data:
    kpi_years = sorted(set(y for d in kpi_data.values() for y in d))
    k_headers = ['KPI'] + [str(y) for y in kpi_years]
    k_rows = []
    for mn in sorted(kpi_data.keys()):
        row_v = [mn.replace('_', ' ').title()]
        for y in kpi_years:
            v = kpi_data[mn].get(y)
            if v is not None:
                if abs(v) >= 1000:
                    row_v.append('{:,.0f}'.format(v))
                else:
                    row_v.append('{:,.2f}'.format(v))
            else:
                row_v.append('-')
        k_rows.append(row_v)
    try:
        display(HTML(html_table(k_headers, k_rows, 'Production KPIs')))
    except Exception:
        print('KPIs: ' + str(len(kpi_data)) + ' metrics')
else:
    print('No production KPI data')

# ── World Aluminium Production (hardcoded industry context) ─────────────────────
world_prod = {
    'China': 41.0, 'India': 4.1, 'Russia (Rusal)': 3.8,
    'Canada': 3.1, 'UAE': 2.7, 'Australia': 1.6,
    'Bahrain': 1.5, 'Norway': 1.4, 'USA': 0.9, 'Other': 8.9
}

fig = go.Figure(go.Pie(
    labels=list(world_prod.keys()),
    values=list(world_prod.values()),
    hole=0.45,
    textinfo='label+percent',
    marker=dict(colors=[
        '#e15759', '#f28e2b', '#4e79a7', '#59a14f', '#76b7b2',
        '#edc948', '#b07aa1', '#ff9da7', '#9c755f', '#bab0ac']),
    textfont=dict(size=10)))

fig.update_layout(
    title='<b>World Aluminium Production (Mt, 2024 est.)</b>',
    height=400, paper_bgcolor='white',
    font=dict(family='Arial', size=11),
    annotations=[dict(text='{:.0f} Mt'.format(sum(world_prod.values())),
                       x=0.5, y=0.5, font_size=16, showarrow=False)])
add_fig(fig)

---
## Section 3: Revenue Analysis

In [ ]:
# ── REVENUE SEGMENTS ─────────────────────────────────────────────────────────────
seg_data = {}
for row in D['segs']:
    key = (row['segment_name'], row['year'])
    if key not in seg_data:
        seg_data[key] = {}
    seg_data[key][row['metric']] = row['value']

# Build segment revenue by year
seg_rev = {}  # {segment: {year: value}}
for (seg, yr), metrics in seg_data.items():
    rev_v = metrics.get('revenue', metrics.get('sales', 0)) or 0
    if rev_v:
        if seg not in seg_rev:
            seg_rev[seg] = {}
        seg_rev[seg][yr] = rev_v

if seg_rev:
    seg_names = sorted(seg_rev.keys())
    seg_years = sorted(set(y for d in seg_rev.values() for y in d))
    seg_colors = [C['rev'], C['ebitda'], C['s1'], C['s2'], C['s3'], C['ni'], C['neutral']]

    fig = go.Figure()
    for i, seg in enumerate(seg_names):
        vals = [M(seg_rev[seg].get(y, 0)) for y in seg_years]
        fig.add_trace(go.Bar(
            x=seg_years, y=vals, name=seg,
            marker_color=seg_colors[i % len(seg_colors)]))
    fig.update_layout(barmode='stack')
    vline(fig)
    add_fig(sty(fig, 'Revenue by Segment ($M)', 420))
else:
    print('No segment data available')

# ── TOTAL REVENUE + YoY GROWTH ──────────────────────────────────────────────────
yrs = ay()
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Revenue ($M)', 'YoY Growth (%)'])

rev_vals = [M(gi(y, 'revenue')) for y in yrs]
fig.add_trace(go.Bar(
    x=yrs, y=rev_vals, name='Revenue',
    marker_color=[C['hist'] if y <= LAST_HIST else C['fcst'] for y in yrs]),
    row=1, col=1)

# YoY growth
growth = []
for idx_y, y in enumerate(yrs):
    if idx_y == 0:
        growth.append(0)
    else:
        prev = gi(yrs[idx_y - 1], 'revenue')
        curr = gi(y, 'revenue')
        growth.append(pct(curr - prev, abs(prev)) if prev else 0)

fig.add_trace(go.Bar(
    x=yrs, y=growth, name='YoY %',
    marker_color=[C['pos'] if g >= 0 else C['neg'] for g in growth]),
    row=1, col=2)

vline(fig)
add_fig(sty(fig, 'Revenue: History + Forecast', 400))

---
## Section 4: Income Statement

In [ ]:
# ── FULL IS TABLE ────────────────────────────────────────────────────────────────
is_metrics = [
    ('Revenue', 'revenue', 'is'),
    ('COGS', 'cogs', 'is'),
    ('Gross Profit', 'gross_profit', 'is'),
    ('SG&A', 'sga', 'is'),
    ('Distribution', 'distribution_expenses', 'is'),
    ('D&A', 'total_da', 'is'),
    ('EBITDA', 'ebitda', 'is'),
    ('EBIT', 'ebit', 'is'),
    ('Interest Expense', 'interest_expense', 'is'),
    ('Interest Income', 'interest_income', 'is'),
    ('Earnings from Investees', 'earnings_from_investees', 'is'),
    ('EBT', 'ebt', 'is'),
    ('Tax Expense', 'tax_expense', 'is'),
    ('Net Income', 'net_income', 'is'),
]

# Select years for table
table_years = [y for y in ay() if y >= max(HIST_START, 2020)]
is_headers = ['Metric ($M)'] + [str(y) + ('*' if y > LAST_HIST else '') for y in table_years]
is_rows = []
bold_metrics = {'Revenue', 'Gross Profit', 'EBITDA', 'EBIT', 'EBT', 'Net Income'}

for label, metric, stmt in is_metrics:
    row_v = []
    if label in bold_metrics:
        row_v.append('<b>' + label + '</b>')
    else:
        row_v.append(label)
    for y in table_years:
        v = gi(y, metric, stmt)
        row_v.append('{:,.0f}'.format(M(v)))
    is_rows.append(row_v)

# Margin rows separator
is_rows.append(['---'] + ['---'] * len(table_years))
for label, num_m, den_m in [
    ('Gross Margin', 'gross_profit', 'revenue'),
    ('EBITDA Margin', 'ebitda', 'revenue'),
    ('EBIT Margin', 'ebit', 'revenue'),
    ('Net Margin', 'net_income', 'revenue'),
]:
    row_v = ['<b>' + label + '</b>']
    for y in table_years:
        n = gi(y, num_m)
        d = gi(y, den_m)
        row_v.append(fmt_pct(pct(n, d)))
    is_rows.append(row_v)

try:
    display(HTML(html_table(is_headers, is_rows, 'Income Statement | * = Forecast')))
except Exception:
    print('IS table: ' + str(len(is_rows)) + ' rows x ' + str(len(table_years)) + ' years')

# ── MARGIN TRENDS CHART ────────────────────────────────────────────────────────
yrs = ay()
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Revenue / EBITDA / Net Income ($M)', 'Margins (%)'],
    vertical_spacing=0.13)

fig.add_trace(go.Bar(
    x=yrs, y=[M(gi(y, 'revenue')) for y in yrs],
    name='Revenue', marker_color='#aec7e8'), row=1, col=1)
fig.add_trace(go.Bar(
    x=yrs, y=[M(gi(y, 'ebitda') or gi(y, 'ebit')) for y in yrs],
    name='EBITDA', marker_color=C['ebitda']), row=1, col=1)
fig.add_trace(go.Bar(
    x=yrs, y=[M(gi(y, 'net_income')) for y in yrs],
    name='NI', marker_color=C['ni']), row=1, col=1)

# Margins
for label, metric, color, dash in [
    ('EBITDA%', 'ebitda', C['ebitda'], 'solid'),
    ('EBIT%', 'ebit', C['s3'], 'dash'),
    ('Net%', 'net_income', C['ni'], 'dot'),
]:
    yrs_m = [y for y in yrs if gi(y, 'revenue')]
    vals = [pct(gi(y, metric), gi(y, 'revenue')) for y in yrs_m]
    fig.add_trace(go.Scatter(
        x=yrs_m, y=vals, name=label,
        line=dict(color=color, width=2, dash=dash)), row=2, col=1)

vline(fig)
fig.update_layout(barmode='group')
add_fig(sty(fig, 'P&L: History + Forecast', 540))

---
## Section 5: Balance Sheet

In [ ]:
# ── BS STRUCTURE CHART ───────────────────────────────────────────────────────────
yrs = ay()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Debt vs Equity ($B)', 'Asset Composition (Latest)'],
    specs=[[{"type": "bar"}, {"type": "pie"}]])

# Debt vs Equity stacked bars
fig.add_trace(go.Bar(
    x=yrs,
    y=[B(abs(gi(y, 'long_term_debt', 'bs')) + abs(gi(y, 'short_term_debt', 'bs'))) for y in yrs],
    name='Total Debt', marker_color=C['debt']), row=1, col=1)
fig.add_trace(go.Bar(
    x=yrs,
    y=[B(gi(y, 'total_equity', 'bs')) for y in yrs],
    name='Equity', marker_color=C['equity']), row=1, col=1)
fig.add_trace(go.Scatter(
    x=yrs,
    y=[B(gi(y, 'cash', 'bs')) for y in yrs],
    name='Cash', line=dict(color=C['cash'], width=2.5)), row=1, col=1)

# Common-size pie for latest year
ly = max(yrs)
bs_items = {}
for lbl, met in [('Cash', 'cash'), ('Receivables', 'accounts_receivable'),
                  ('Inventory', 'inventory'), ('PP&E', 'ppe_net'),
                  ('Investments', 'investments_lt')]:
    val = abs(gi(ly, met, 'bs'))
    if val > 0:
        bs_items[lbl] = val

other_val = abs(gi(ly, 'total_assets', 'bs')) - sum(bs_items.values())
if other_val > 0:
    bs_items['Other'] = other_val

if bs_items:
    pie_colors = [C['cash'], C['rev'], C['ebitda'], C['s3'], C['ni'], C['neutral']]
# Pie chart handled separately (can't mix with subplots)


---
## Section 6: Cash Flow

In [ ]:
# ── CFO / CFI / CFF BARS + FCF ──────────────────────────────────────────────────
yrs = ay()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['CFO / CFI / CFF ($M)', 'Free Cash Flow ($M)'])

cfo = [M(gi(y, 'cfo_total', 'cf')) for y in yrs]
cfi = [M(gi(y, 'cfi_total', 'cf')) for y in yrs]
cff = [M(gi(y, 'cff_total', 'cf')) for y in yrs]
capex_v = [M(abs(gi(y, 'capex', 'cf') or gi(y, 'cfi_capex', 'cf'))) for y in yrs]
fcf = [o - c for o, c in zip(cfo, capex_v)]

fig.add_trace(go.Bar(x=yrs, y=cfo, name='CFO', marker_color=C['cfo']), row=1, col=1)
fig.add_trace(go.Bar(x=yrs, y=cfi, name='CFI', marker_color=C['cfi']), row=1, col=1)
fig.add_trace(go.Bar(x=yrs, y=cff, name='CFF', marker_color=C['cff']), row=1, col=1)

fig.add_trace(go.Bar(
    x=yrs, y=fcf, name='FCF',
    marker_color=[C['pos'] if v >= 0 else C['neg'] for v in fcf]), row=1, col=2)

fig.update_layout(barmode='group')
vline(fig)
add_fig(sty(fig, 'Cash Flow Analysis', 400))

# ── CASH CONVERSION CYCLE ──────────────────────────────────────────────────────
def calc_ccc(yr):
    rev_v = gi(yr, 'revenue')
    cogs_v = abs(gi(yr, 'cogs'))
    ar = gi(yr, 'accounts_receivable', 'bs')
    inv = gi(yr, 'inventory', 'bs')
    ap = gi(yr, 'accounts_payable', 'bs')
    dso = sdiv(ar, rev_v) * 365 if rev_v else 0
    dio = sdiv(inv, cogs_v) * 365 if cogs_v else 0
    dpo = sdiv(ap, cogs_v) * 365 if cogs_v else 0
    return dso, dio, dpo, dso + dio - dpo

ccc_yrs = [y for y in yrs if gi(y, 'revenue')]
if ccc_yrs:
    dso_v = [calc_ccc(y)[0] for y in ccc_yrs]
    dio_v = [calc_ccc(y)[1] for y in ccc_yrs]
    dpo_v = [calc_ccc(y)[2] for y in ccc_yrs]
    ccc_v = [calc_ccc(y)[3] for y in ccc_yrs]

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=ccc_yrs, y=dso_v, name='DSO',
                              line=dict(color=C['rev'], width=2)))
    fig2.add_trace(go.Scatter(x=ccc_yrs, y=dio_v, name='DIO',
                              line=dict(color=C['ebitda'], width=2)))
    fig2.add_trace(go.Scatter(x=ccc_yrs, y=dpo_v, name='DPO',
                              line=dict(color=C['s1'], width=2)))
    fig2.add_trace(go.Scatter(x=ccc_yrs, y=ccc_v, name='CCC',
                              line=dict(color=C['header'], width=3)))
    vline(fig2)
    add_fig(sty(fig2, 'Cash Conversion Cycle (Days)', 380))

---
## Section 7: Debt Analysis

In [ ]:
# ── DEBT INSTRUMENTS TABLE ───────────────────────────────────────────────────────
di = D['di']
if di:
    di_headers = ['Instrument', 'Type', 'Currency', 'Rate %', 'Maturity', 'Balance ($M)']
    di_rows = []
    type_totals = {}
    ccy_totals = {}

    for d in di:
        name = d.get('instrument_name', d.get('instrument_id', '-'))
        itype = d.get('instrument_type', '-')
        ccy = d.get('currency', 'USD')
        rate = d.get('coupon_rate', d.get('interest_rate', 0)) or 0
        mat = d.get('maturity_year', d.get('maturity_date', '-'))
        bal = d.get('opening_balance', 0) or 0

        di_rows.append([
            name, itype, ccy,
            '{:.2f}'.format(rate * 100 if abs(rate) < 1 else rate),
            str(mat),
            '{:,.0f}'.format(M(bal))])

        type_totals[itype] = type_totals.get(itype, 0) + bal
        ccy_totals[ccy] = ccy_totals.get(ccy, 0) + bal

    try:
        display(HTML(html_table(di_headers, di_rows, 'Debt Instruments')))
    except Exception:
        print('Debt instruments: ' + str(len(di)))

    # ── MATURITY SCHEDULE ───────────────────────────────────────────────────────
    mat_data = {}
    for d in di:
        my = d.get('maturity_year')
        bal = d.get('opening_balance', 0) or 0
        if my and bal:
            mat_data[my] = mat_data.get(my, 0) + bal

    if mat_data:
        mat_years = sorted(mat_data.keys())
        fig = go.Figure(go.Bar(
            x=mat_years,
            y=[M(mat_data[y]) for y in mat_years],
            marker_color=C['debt'],
            text=['{:,.0f}'.format(M(mat_data[y])) for y in mat_years],
            textposition='outside'))
        add_fig(sty(fig, 'Debt Maturity Schedule ($M)', 380))

    # ── CURRENCY DONUT ──────────────────────────────────────────────────────────
    if len(ccy_totals) > 1:
        fig = go.Figure(go.Pie(
            labels=list(ccy_totals.keys()),
            values=[M(v) for v in ccy_totals.values()],
            hole=0.45, textinfo='label+percent+value',
            marker=dict(colors=[C['rev'], C['s1'], C['ebitda'], C['s2']])))
        add_fig(sty(fig, 'Debt by Currency ($M)', 380))
    elif ccy_totals:
        print('Single currency debt: ' + list(ccy_totals.keys())[0])
else:
    print('No debt instrument data')

# ── ND/EBITDA + STRESS OVERLAY ──────────────────────────────────────────────────
yrs = ay()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Net Debt / EBITDA', 'Interest Coverage (EBITDA/Int)'])

def nd_eb_calc(yr):
    td = abs(gi(yr, 'long_term_debt', 'bs')) + abs(gi(yr, 'short_term_debt', 'bs'))
    nd = td - gi(yr, 'cash', 'bs')
    eb = gi(yr, 'ebitda') or gi(yr, 'ebit') or 1
    return sdiv(nd, eb)

fig.add_trace(go.Scatter(
    x=yrs, y=[nd_eb_calc(y) for y in yrs],
    name='Base', line=dict(color=C['base'], width=2.5)), row=1, col=1)

sc_colors = [C['s1'], C['s2'], C['s3']]
for i, (sc, sr) in enumerate(D['stress'].items()):
    if sr and sr.success and i < 3:
        sv_d = getattr(sr, 'stress_values', {}) or {}
        yf_s = sorted(sv_d.keys())
        vals = []
        for y in yf_s:
            s = sv_d[y] if isinstance(sv_d[y], dict) else {}
            d_ = abs(s.get('long_term_debt', 0) or 0) + abs(s.get('short_term_debt', 0) or 0)
            e_ = s.get('ebitda', 0) or 1
            vals.append(sdiv(d_ - (s.get('cash', 0) or 0), e_))
        fig.add_trace(go.Scatter(
            x=yf_s, y=vals, name=sc,
            line=dict(color=sc_colors[i], dash='dot', width=1.5)), row=1, col=1)

fig.add_hline(y=4.5, line_dash='dash', line_color='red', opacity=0.5,
              annotation_text='Covenant', row=1, col=1)

# ICR
icr_vals = []
for y in yrs:
    eb = gi(y, 'ebitda') or gi(y, 'ebit')
    ie = abs(gi(y, 'interest_expense'))
    icr_vals.append(sdiv(eb, ie) if ie else 0)

fig.add_trace(go.Scatter(
    x=yrs, y=icr_vals, name='ICR',
    line=dict(color=C['s3'], width=2)), row=1, col=2)
fig.add_hline(y=2.0, line_dash='dash', line_color='red', opacity=0.5,
              annotation_text='Min ICR', row=1, col=2)

vline(fig)
add_fig(sty(fig, 'Leverage & Coverage', 420))

---
## Section 8: Credit Metrics & Covenant Heatmap

In [ ]:
# ── COMPREHENSIVE RATIO TABLE ────────────────────────────────────────────────────
ratio_defs = [
    ('Gross Margin %', lambda y: pct(gi(y, 'gross_profit'), gi(y, 'revenue')), 'Profitability'),
    ('EBITDA Margin %', lambda y: pct(gi(y, 'ebitda'), gi(y, 'revenue')), 'Profitability'),
    ('EBIT Margin %', lambda y: pct(gi(y, 'ebit'), gi(y, 'revenue')), 'Profitability'),
    ('Net Margin %', lambda y: pct(gi(y, 'net_income'), gi(y, 'revenue')), 'Profitability'),
    ('ROA %', lambda y: pct(gi(y, 'net_income'), gi(y, 'total_assets', 'bs')), 'Profitability'),
    ('ROE %', lambda y: pct(gi(y, 'net_income'), gi(y, 'total_equity', 'bs')), 'Profitability'),
    ('ND/EBITDA', lambda y: sdiv(
        abs(gi(y, 'long_term_debt', 'bs')) + abs(gi(y, 'short_term_debt', 'bs')) - gi(y, 'cash', 'bs'),
        gi(y, 'ebitda') or 1), 'Leverage'),
    ('Debt/Equity', lambda y: sdiv(
        abs(gi(y, 'long_term_debt', 'bs')) + abs(gi(y, 'short_term_debt', 'bs')),
        gi(y, 'total_equity', 'bs') or 1), 'Leverage'),
    ('Debt/Capital', lambda y: (lambda td, eq: sdiv(td, td + eq) if (td + eq) else 0)(
        abs(gi(y, 'long_term_debt', 'bs')) + abs(gi(y, 'short_term_debt', 'bs')),
        gi(y, 'total_equity', 'bs') or 0), 'Leverage'),
    ('EBITDA/Interest', lambda y: sdiv(gi(y, 'ebitda'), abs(gi(y, 'interest_expense')) or 1), 'Coverage'),
    ('EBIT/Interest', lambda y: sdiv(gi(y, 'ebit'), abs(gi(y, 'interest_expense')) or 1), 'Coverage'),
    ('Current Ratio', lambda y: sdiv(gi(y, 'total_ca', 'bs'), gi(y, 'total_cl', 'bs') or 1), 'Liquidity'),
    ('Cash/Debt', lambda y: sdiv(gi(y, 'cash', 'bs'),
        (abs(gi(y, 'long_term_debt', 'bs')) + abs(gi(y, 'short_term_debt', 'bs'))) or 1), 'Liquidity'),
    ('DSO (days)', lambda y: sdiv(gi(y, 'accounts_receivable', 'bs'), gi(y, 'revenue') or 1) * 365, 'Turnover'),
    ('DIO (days)', lambda y: sdiv(gi(y, 'inventory', 'bs'), abs(gi(y, 'cogs')) or 1) * 365, 'Turnover'),
    ('DPO (days)', lambda y: sdiv(gi(y, 'accounts_payable', 'bs'), abs(gi(y, 'cogs')) or 1) * 365, 'Turnover'),
    ('CFO/Debt', lambda y: sdiv(gi(y, 'cfo_total', 'cf'),
        (abs(gi(y, 'long_term_debt', 'bs')) + abs(gi(y, 'short_term_debt', 'bs'))) or 1), 'Cash Flow'),
    ('FCF ($M)', lambda y: M(gi(y, 'cfo_total', 'cf') + (gi(y, 'cfi_capex', 'cf') or gi(y, 'capex', 'cf'))), 'Cash Flow'),
    ('Capex/Rev %', lambda y: pct(abs(gi(y, 'cfi_capex', 'cf') or gi(y, 'capex', 'cf')), gi(y, 'revenue') or 1), 'Cash Flow'),
]

table_yrs = [y for y in ay() if y >= max(HIST_START, 2019)]
r_headers = ['Metric'] + [str(y) + ('*' if y > LAST_HIST else '') for y in table_yrs]

r_rows = []
current_section = ''
for item in ratio_defs:
    label, func, section = item[0], item[1], item[2]
    if section != current_section:
        r_rows.append(['<b>' + section.upper() + '</b>'] + [''] * len(table_yrs))
        current_section = section
    row_v = [label]
    for y in table_yrs:
        try:
            val = func(y)
        except Exception:
            val = 0
        if '%' in label:
            row_v.append(fmt_pct(val))
        elif 'days' in label.lower():
            row_v.append('{:.0f}'.format(val))
        elif 'FCF' in label:
            row_v.append('{:,.0f}'.format(val))
        else:
            row_v.append('{:.2f}x'.format(val))
    r_rows.append(row_v)

try:
    display(HTML(html_table(r_headers, r_rows, 'Comprehensive Credit Metrics')))
except Exception:
    print('Ratio table: ' + str(len(r_rows)) + ' rows')

# ── COVENANT HEATMAP ────────────────────────────────────────────────────────────
cov_metrics = [
    ('ND/EBITDA', lambda y: sdiv(
        abs(gi(y, 'long_term_debt', 'bs')) + abs(gi(y, 'short_term_debt', 'bs')) - gi(y, 'cash', 'bs'),
        gi(y, 'ebitda') or 1)),
    ('EBITDA/Interest', lambda y: sdiv(gi(y, 'ebitda'), abs(gi(y, 'interest_expense')) or 1)),
    ('Current Ratio', lambda y: sdiv(gi(y, 'total_ca', 'bs'), gi(y, 'total_cl', 'bs') or 1)),
]

cov_yrs = list(fy())
if cov_yrs:
    z_data = []
    y_labels = []
    x_labels = [str(y) for y in cov_yrs]
    cov_text = []

    for label, func in cov_metrics:
        row_z = []
        row_t = []
        for y in cov_yrs:
            try:
                val = func(y)
            except Exception:
                val = 0
            # Normalize: green=good, red=bad
            if label == 'ND/EBITDA':
                score = max(0, min(1, 1 - val / 6.0))
            else:
                score = max(0, min(1, val / 5.0))
            row_z.append(score)
            row_t.append('{:.2f}'.format(val))
        z_data.append(row_z)
        y_labels.append(label)
        cov_text.append(row_t)

    fig = go.Figure(go.Heatmap(
        z=z_data, x=x_labels, y=y_labels,
        text=cov_text, texttemplate='%{text}',
        colorscale=[[0, '#e15759'], [0.5, '#f28e2b'], [1, '#59a14f']],
        showscale=False))

    add_fig(sty(fig, 'Covenant Heatmap (Forecast Years)', 250))

---
## Section 9: Credit Rating

In [ ]:
# ── RATING TRAJECTORY ────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Rating Trajectory', 'Sub-Scores (Latest Year)'],
    column_widths=[0.6, 0.4])

ry = sorted(D['ratings'])
if ry:
    # Main trajectory
    fig.add_trace(go.Scatter(
        x=ry,
        y=[rn(D['ratings'][y]['rating']) for y in ry],
        mode='lines+markers+text',
        text=[D['ratings'][y]['rating'] for y in ry],
        textposition='top center',
        line=dict(color=C['base'], width=2.5),
        marker=dict(size=9, color=C['base']),
        name='Base Rating'), row=1, col=1)

    # IG threshold
    ig_level = rn('BBB-')
    fig.add_hline(y=ig_level, line_dash='dash', line_color='green',
                  opacity=0.5, annotation_text='IG Threshold', row=1, col=1)

    fig.update_yaxes(
        tickvals=list(range(1, len(RATING_SCALE) + 1)),
        ticktext=list(reversed(RATING_SCALE)),
        row=1, col=1)

    # Stress rating overlay
    sc_colors = [C['s1'], C['s2'], C['s3']]
    for i, (sc, sr) in enumerate(D['stress'].items()):
        if sr and sr.success and i < 3:
            sv_d = getattr(sr, 'stress_values', {}) or {}
            stress_ratings = {}
            for yr_s in sorted(sv_d.keys()):
                try:
                    sv_dict = sv_d[yr_s]
                    if isinstance(sv_dict, dict):
                        class _NS:
                            pass
                        ns = _NS()
                        for k_attr, v_attr in sv_dict.items():
                            setattr(ns, k_attr, v_attr)
                        for attr in ['total_ca', 'total_cl', 'cfo_total', 'cfi_capex']:
                            if not hasattr(ns, attr):
                                setattr(ns, attr, 0)
                        m = CreditMetrics.from_year_state(ns, yr_s)
                        r = eng.calculate(m)
                        stress_ratings[yr_s] = r['rating']
                except Exception:
                    pass
            if stress_ratings:
                sy = sorted(stress_ratings.keys())
                fig.add_trace(go.Scatter(
                    x=sy,
                    y=[rn(stress_ratings[y]) for y in sy],
                    name=sc,
                    line=dict(color=sc_colors[i], dash='dot', width=1.5),
                    mode='lines+markers',
                    marker=dict(size=6)), row=1, col=1)

    # Sub-scores bar for latest year
    latest_r = D['ratings'].get(max(ry), {})
    sub = latest_r.get('sub_scores', {})
    if sub:
        cats = list(sub.keys())
        vals = [sub[k] for k in cats]
        bar_colors = [C['pos'] if v >= 50 else C['s2'] if v >= 30 else C['neg'] for v in vals]
        fig.add_trace(go.Bar(
            y=[c_name.title() for c_name in cats],
            x=vals,
            orientation='h',
            marker_color=bar_colors,
            text=['{:.0f}'.format(v) for v in vals],
            textposition='outside',
            showlegend=False,
            name='Sub-scores'), row=1, col=2)
        fig.update_xaxes(range=[0, 100], title_text='Score (0-100)', row=1, col=2)

add_fig(sty(fig, 'Credit Rating Analysis', 450))

# ── RATING SUMMARY TABLE ───────────────────────────────────────────────────────
if ry:
    rt_headers = ['Year', 'Rating', 'Score', 'Leverage', 'Coverage', 'Profitability', 'Liquidity']
    rt_rows = []
    for y in ry:
        rd = D['ratings'][y]
        ss = rd.get('sub_scores', {})
        rt_rows.append([
            str(y), rd.get('rating', '-'), '{:.1f}'.format(rd.get('score', 0)),
            '{:.0f}'.format(ss.get('leverage', 0)),
            '{:.0f}'.format(ss.get('coverage', 0)),
            '{:.0f}'.format(ss.get('profitability', 0)),
            '{:.0f}'.format(ss.get('liquidity', 0))])
    try:
        display(HTML(html_table(rt_headers, rt_rows, 'Rating Details by Year')))
    except Exception:
        for r_row in rt_rows:
            print(' | '.join(r_row))

---
## Section 10: Stress Testing

In [ ]:
# ── 4-PANEL STRESS COMPARISON ────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Revenue ($M)', 'EBITDA Margin %', 'Net Debt / EBITDA', 'Net Income ($M)'],
    vertical_spacing=0.15)

yf = fy()
sc_colors = [C['s1'], C['s2'], C['s3']]

panels = [
    ('revenue', 1, 1),
    ('ebitda_pct', 1, 2),
    ('nd_ebitda', 2, 1),
    ('net_income', 2, 2),
]

for attr, r, c in panels:
    # Base line
    if attr == 'ebitda_pct':
        base_vals = [pct(sv_get(y, 'ebitda'), sv_get(y, 'revenue') or 1) for y in yf]
    elif attr == 'nd_ebitda':
        base_vals = [sv_nd_ebitda(y) for y in yf]
    else:
        base_vals = [M(sv_get(y, attr)) for y in yf]

    fig.add_trace(go.Scatter(
        x=yf, y=base_vals, name='Base',
        line=dict(color=C['base'], width=2.5),
        showlegend=(r == 1 and c == 1)), row=r, col=c)

    # Stress lines
    for i, (sc, sr) in enumerate(D['stress'].items()):
        if sr and sr.success and i < 3:
            if attr == 'ebitda_pct':
                s_vals = [pct(sv_get(y, 'ebitda', sr), sv_get(y, 'revenue', sr) or 1) for y in yf]
            elif attr == 'nd_ebitda':
                s_vals = [sv_nd_ebitda(y, sr) for y in yf]
            else:
                s_vals = [M(sv_get(y, attr, sr)) for y in yf]

            fig.add_trace(go.Scatter(
                x=yf, y=s_vals, name=sc,
                line=dict(color=sc_colors[i], dash='dot', width=1.5),
                showlegend=(r == 1 and c == 1)), row=r, col=c)

add_fig(sty(fig, 'Stress Testing: 4-Panel Comparison', 540))

# ── TORNADO CHART ───────────────────────────────────────────────────────────────
yr0 = min(yf) if yf else LAST_HIST + 1
base_ni = M(sv_get(yr0, 'net_income'))
base_eb = M(sv_get(yr0, 'ebitda'))

tornado_data = []
for sc, sr in D['stress'].items():
    if sr and sr.success:
        s_ni = M(sv_get(yr0, 'net_income', sr))
        s_eb = M(sv_get(yr0, 'ebitda', sr))
        tornado_data.append((sc, s_ni - base_ni, s_eb - base_eb))

if tornado_data:
    tornado_data.sort(key=lambda x: abs(x[1]), reverse=True)
    sc_names = [t[0] for t in tornado_data]
    ni_deltas = [t[1] for t in tornado_data]
    eb_deltas = [t[2] for t in tornado_data]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=sc_names, x=ni_deltas, orientation='h',
        name='NI Impact ($M)',
        marker_color=[C['neg'] if v < 0 else C['pos'] for v in ni_deltas],
        text=['{:+,.0f}'.format(v) for v in ni_deltas],
        textposition='outside'))
    fig.add_trace(go.Bar(
        y=sc_names, x=eb_deltas, orientation='h',
        name='EBITDA Impact ($M)',
        marker_color=[C['s2'] if v < 0 else C['ebitda'] for v in eb_deltas],
        text=['{:+,.0f}'.format(v) for v in eb_deltas],
        textposition='outside',
        opacity=0.6))
    fig.update_layout(barmode='group')
    add_fig(sty(fig, 'Tornado: Stress Impact ($M, ' + str(yr0) + ')', 350))

# ── SENSITIVITY TABLE ───────────────────────────────────────────────────────────
if D['stress']:
    sens_headers = ['Scenario', 'Year', 'Revenue ($M)', 'EBITDA ($M)', 'NI ($M)', 'ND/EBITDA']
    sens_rows = []
    # Base rows
    for y in yf[:3]:
        sens_rows.append([
            'BASE', str(y),
            '{:,.0f}'.format(M(sv_get(y, 'revenue'))),
            '{:,.0f}'.format(M(sv_get(y, 'ebitda'))),
            '{:,.0f}'.format(M(sv_get(y, 'net_income'))),
            '{:.2f}x'.format(sv_nd_ebitda(y))])

    for sc, sr in D['stress'].items():
        if sr and sr.success:
            for y in yf[:3]:
                sens_rows.append([
                    sc, str(y),
                    '{:,.0f}'.format(M(sv_get(y, 'revenue', sr))),
                    '{:,.0f}'.format(M(sv_get(y, 'ebitda', sr))),
                    '{:,.0f}'.format(M(sv_get(y, 'net_income', sr))),
                    '{:.2f}x'.format(sv_nd_ebitda(y, sr))])

    try:
        display(HTML(html_table(sens_headers, sens_rows, 'Stress Sensitivity Table')))
    except Exception:
        print('Sensitivity table: ' + str(len(sens_rows)) + ' rows')

---
## Section 11: Export to Standalone HTML

In [ ]:
# ── EXPORT ALL TO STANDALONE HTML ────────────────────────────────────────────────
if EXPORT_HTML:
    from datetime import datetime

    out = P('companies') / COMPANY_ID / 'outputs'
    out.mkdir(exist_ok=True, parents=True)
    hp = out / (COMPANY_ID + '_dashboard.html')

    yr0 = min(fy()) if fy() else LAST_HIST + 1
    now = datetime.now().strftime('%Y-%m-%d %H:%M')

    # Gather KPI values
    s0 = D['fc'].get(yr0)
    rev_v = getattr(s0, 'revenue', 0) if s0 else 0
    eb_v = getattr(s0, 'ebitda', 0) if s0 else 0
    ni_v = getattr(s0, 'net_income', 0) if s0 else 0
    rat_v = D['ratings'].get(yr0, {}).get('rating', '?')

    parts = []
    parts.append('<!DOCTYPE html><html><head><meta charset="utf-8">')
    parts.append('<title>' + COMPANY_ID.upper() + ' Financial Dashboard</title>')
    parts.append('<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>')
    parts.append('<style>')
    parts.append('body { font-family: Arial, sans-serif; margin: 24px; background: #f5f5f5; }')
    parts.append('h1 { color: #343a40; border-bottom: 3px solid #007bff; padding-bottom: 8px; }')
    parts.append('.section { background: white; border-radius: 8px; padding: 18px; margin: 16px 0;')
    parts.append('  box-shadow: 0 2px 4px rgba(0,0,0,.08); }')
    parts.append('.kpi { display: flex; gap: 16px; flex-wrap: wrap; margin: 16px 0; }')
    parts.append('.kpi-card { background: #fff; border: 1px solid #dee2e6; border-radius: 8px;')
    parts.append('  padding: 14px 20px; min-width: 140px; }')
    parts.append('.kpi-label { font-size: 11px; color: #888; }')
    parts.append('.kpi-value { font-size: 22px; font-weight: 700; color: #343a40; }')
    parts.append('</style></head><body>')
    parts.append('<h1>' + COMPANY_ID.upper() + ' -- Financial Dashboard</h1>')
    parts.append('<p style="color:#888">Generated: ' + now + ' | Charts: ' + str(len(FIGS)) + '</p>')
    parts.append('<div class="kpi">')
    parts.append('<div class="kpi-card"><div class="kpi-label">Revenue '
                 + str(yr0) + '</div><div class="kpi-value">$'
                 + '{:,.0f}'.format(M(rev_v)) + 'M</div></div>')
    parts.append('<div class="kpi-card"><div class="kpi-label">EBITDA</div><div class="kpi-value">$'
                 + '{:,.0f}'.format(M(eb_v)) + 'M ('
                 + '{:.1f}'.format(pct(eb_v, rev_v)) + '%)</div></div>')
    parts.append('<div class="kpi-card"><div class="kpi-label">Net Income</div><div class="kpi-value">$'
                 + '{:,.0f}'.format(M(ni_v)) + 'M</div></div>')
    parts.append('<div class="kpi-card"><div class="kpi-label">Rating</div><div class="kpi-value">'
                 + rat_v + '</div></div>')
    parts.append('</div>')

    section_titles = [
        'Macro Environment', 'World Aluminium Production',
        'Revenue Segments', 'Revenue Trend',
        'Income Statement (Margins)', 'Balance Sheet',
        'Cash Flow', 'Cash Conversion Cycle',
        'Debt Maturity', 'Debt by Currency',
        'Leverage & Coverage', 'Covenant Heatmap',
        'Credit Rating', 'Stress 4-Panel',
        'Stress Tornado',
    ]

    for i, fig_item in enumerate(FIGS):
        title = section_titles[i] if i < len(section_titles) else 'Chart ' + str(i + 1)
        parts.append('<div class="section"><h2>' + title + '</h2>')
        try:
            parts.append(fig_item.to_html(full_html=False, include_plotlyjs=False))
        except Exception:
            parts.append('<p>Chart rendering error</p>')
        parts.append('</div>')

    parts.append('</body></html>')

    hp.write_text('\n'.join(parts), encoding='utf-8')
    size_kb = hp.stat().st_size // 1024
    print('Dashboard exported: ' + str(hp))
    print('Size: ' + str(size_kb) + ' KB | Charts: ' + str(len(FIGS)))
else:
    print('Export disabled (EXPORT_HTML=False)')